In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

class CrashAnalysis:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path, low_memory=False)
        self.encoders = {}
        self.model = None

    def create_heatmap(self, output_file='nyc_collision_heatmap.html'):
        print('Generating heatmap, please wait...')
        m = folium.Map(location=[40.7128, -74.0060], zoom_start=11)
        locations = self.df[['LATITUDE', 'LONGITUDE']].dropna()
        heat_data = [[row['LATITUDE'], row['LONGITUDE']] for _, row in locations.iterrows()]
        HeatMap(heat_data).add_to(m)
        m.save(output_file)
        import os
        print(f'Heatmap created successfully. You can view it at: {os.path.abspath(output_file)}')

    def preprocess_data(self):
        self.df['InjuryOccurred'] = (self.df['NUMBER OF PERSONS INJURED'] > 0).astype(int)
        self.df['CRASH HOUR'] = pd.to_datetime(self.df['CRASH TIME'], format='%H:%M', errors='coerce').dt.hour
        self.df['DAY_OF_WEEK'] = pd.to_datetime(self.df['CRASH DATE']).dt.dayofweek
        self.df['Total Injuries'] = self.df['NUMBER OF PERSONS INJURED']
        self.df_model = self.df[['BOROUGH', 'VEHICLE TYPE CODE 1', 'CONTRIBUTING FACTOR VEHICLE 1', 'CRASH HOUR', 'DAY_OF_WEEK', 'InjuryOccurred']].dropna()

        for col in ['BOROUGH', 'VEHICLE TYPE CODE 1', 'CONTRIBUTING FACTOR VEHICLE 1']:
            le = LabelEncoder()
            self.df_model[col] = le.fit_transform(self.df_model[col])
            self.encoders[col] = le

    def train_model(self):
        print('Model training in progress, please wait...')
        X = self.df_model.drop('InjuryOccurred', axis=1)
        y = self.df_model['InjuryOccurred']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        self.model.fit(X_train, y_train)
        print('Model trained successfully.')

    def predict_injury(self, borough, vehicle_type, contributing_factor, crash_hour, day_of_week):
        borough_mapping = {0: 'MANHATTAN', 1: 'BROOKLYN', 2: 'QUEENS', 3: 'BRONX', 4: 'STATEN ISLAND'}
        vehicle_mapping = {0: 'Sedan', 1: 'Taxi', 2: 'Bicycle', 3: 'SUV', 4: 'Pickup Truck'}
        factor_mapping = {0: 'Aggressive Driving/Road Rage', 1: 'Pavement Slippery', 2: 'Unspecified', 3: 'Following Too Closely', 4: 'Passing Too Closely', 5: 'Unsafe Speed'}

        try:
            borough = borough_mapping[borough]
            vehicle_type = vehicle_mapping[vehicle_type]
            contributing_factor = factor_mapping[contributing_factor]
        except KeyError:
            print("Invalid entry. Please ensure your selections are within the specified ranges.")
            return None
        if self.model is None:
            print('Model has not been trained yet. Please train the model first.')
            return None

        input_data = pd.DataFrame([{ 'BOROUGH': borough, 'VEHICLE TYPE CODE 1': vehicle_type, 'CONTRIBUTING FACTOR VEHICLE 1': contributing_factor, 'CRASH HOUR': crash_hour, 'DAY_OF_WEEK': day_of_week }])

        for col in ['BOROUGH', 'VEHICLE TYPE CODE 1', 'CONTRIBUTING FACTOR VEHICLE 1']:
            try:
                input_data[col] = self.encoders[col].transform(input_data[col])
            except ValueError:
                print(f"Invalid entry for {col}. Please enter a valid option.")
                return None

        prediction = self.model.predict(input_data)[0]
        probability = self.model.predict_proba(input_data)[0][1] if prediction == 1 else self.model.predict_proba(input_data)[0][0]
        confidence = '{:.2%}'.format(probability)
        return f'{"Injury" if prediction == 1 else "No Injury"} ({confidence} confidence)'

    def visualize_data(self):
        while True:
            print("\nVisual Options:")
            print("1. Injuries Over Time")
            print("2. Crashes by Hour of the Day")
            print("3. Back to Main Menu")
            choice = input("Choose a visual to create: ")

            if choice == "1":
                print('Creating Injuries Over Time visualization, please wait...')
                self.df['CRASH DATE'] = pd.to_datetime(self.df['CRASH DATE'], errors='coerce')
                injuries_per_day = self.df.groupby('CRASH DATE')['Total Injuries'].sum()
                plt.figure(figsize=(12, 6))
                injuries_per_day.sort_index().plot()
                plt.title("Injuries Over Time")
                plt.xlabel("Date")
                plt.ylabel("Total Number of Injuries")
                plt.grid(True)
                plt.xticks(rotation=45)
                plt.show()
            elif choice == "2":
                print('Creating Crashes by Hour of the Day visualization, please wait...')
                plt.figure(figsize=(12, 6))
                sns.countplot(data=self.df, x='CRASH HOUR', hue='InjuryOccurred')
                plt.title("Crashes by Hour of the Day")
                plt.xlabel("Hour")
                plt.ylabel("Number of Crashes")
                plt.xticks(rotation=45)
                plt.show()
            elif choice == "3":
                break
            else:
                print("Invalid choice. Please select a valid option.")

    def view_model_metrics(self):
        if self.model is None:
            print("Model has not been trained yet. Please train the model first.")
        else:
            from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
            X = self.df_model.drop('InjuryOccurred', axis=1)
            y = self.df_model['InjuryOccurred']
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            y_pred = self.model.predict(X_test)
            print("Model Performance Metrics:")
            if accuracy_score(y_test, y_pred) < 0.7:
                print("The model's performance could be improved with a dataset containing more injury cases, allowing the model to learn better patterns and make more accurate predictions.")
            print("Accuracy: {:.2%}".format(accuracy_score(y_test, y_pred)))
            print("Precision: {:.2%}".format(precision_score(y_test, y_pred)))
            print("Recall: {:.2%}".format(recall_score(y_test, y_pred)))
            print("F1 Score: {:.2%}".format(f1_score(y_test, y_pred)))

if __name__ == "__main__":
    file_path = input("Enter the CSV file path: ")
    print('Data processing in progress, please wait...')
    analysis = CrashAnalysis(file_path)
    analysis.preprocess_data()

    while True:
        print("\nOptions:")
        print("1. Create Heatmap")
        print("2. Train Model")
        print("3. Predict Injury")
        print("4. Visualize Data")
        print("5. View Model Metrics")
        print("6. Exit")
        choice = input("Choose an option: ")

        if choice == "1":
            analysis.create_heatmap()
        elif choice == "2":
            analysis.train_model()
        elif choice == "3":
            if analysis.model is None:
                print("Model has not been trained yet. Please train the model first.")
            else:
                print("Borough options:")
                print("0: MANHATTAN")
                print("1: BROOKLYN")
                print("2: QUEENS")
                print("3: BRONX")
                print("4: STATEN ISLAND")
                borough = int(input("Enter borough (0-4): "))
                print("Vehicle type options:")
                print("0: Sedan")
                print("1: Taxi")
                print("2: Bicycle")
                print("3: SUV")
                print("4: Pickup Truck")
                vehicle_type = int(input("Enter vehicle type (0-4): "))
                print("Contributing factor options:")
                print("0: Aggressive Driving/Road Rage")
                print("1: Pavement Slippery")
                print("2: Unspecified")
                print("3: Following Too Closely")
                print("4: Passing Too Closely")
                print("5: Unsafe Speed")
                contributing_factor = int(input("Enter contributing factor (0-5): "))
                crash_hour = int(input("Enter crash hour (0-23): "))
                day_of_week = int(input("Enter day of the week (0-6): "))
                result = analysis.predict_injury(borough, vehicle_type, contributing_factor, crash_hour, day_of_week)
                print("Prediction:", result)
        elif choice == "4":
            analysis.visualize_data()
        elif choice == "5":
            analysis.view_model_metrics()
        elif choice == "6":
            break
        else:
            print("Invalid choice. Please try again.")


Enter the CSV file path:  ./Motor_Vehicle_Collisions_-_Crashes.csv


Data processing in progress, please wait...

Options:
1. Create Heatmap
2. Train Model
3. Predict Injury
4. Visualize Data
5. View Model Metrics
6. Exit


Choose an option:  2


Model training in progress, please wait...
Model trained successfully.

Options:
1. Create Heatmap
2. Train Model
3. Predict Injury
4. Visualize Data
5. View Model Metrics
6. Exit


Choose an option:  3


...

Vehicle type options:
0: Sedan
1: Taxi
2: Bicycle
3: SUV
4: Pickup Truck


Enter vehicle type (0-4):  0


Contributing factor options:
0: Aggressive Driving/Road Rage
1: Pavement Slippery
2: Unspecified
3: Following Too Closely
4: Passing Too Closely
5: Unsafe Speed


Enter contributing factor (0-5):  5
Enter crash hour (0-23):  23
Enter day of the week (0-6):  5


Prediction: No Injury (59.53% confidence)

Options:
1. Create Heatmap
2. Train Model
3. Predict Injury
4. Visualize Data
5. View Model Metrics
6. Exit
